# 戦略4: バリュートラップ回避モデル

**作成日**: 2026-02-21

**目的**: 割安株の中から、バリュートラップ（株価停滞銘柄）を回避するフィルタの有効性を検証

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
import json
import warnings
warnings.filterwarnings('ignore')

print("ライブラリインポート完了")

ライブラリインポート完了


## 1. データ読み込み

In [2]:
PROJECT_ROOT = Path(r'C:\Users\yongr\claude project\workspace')

# 価格データ
print("価格データ読み込み中...")
df_price = pd.read_parquet(PROJECT_ROOT / 'data/curated/jquants/prices/daily_quotes_all.parquet')
df_price['date'] = pd.to_datetime(df_price['date'])
df_price = df_price[df_price['date'] >= '2017-01-01'].copy()
print(f"価格データ: {len(df_price):,} 行")

# 財務データ
print("財務データ読み込み中...")
df_fin = pd.read_parquet(PROJECT_ROOT / 'data/curated/jquants/financials/statements_all.parquet')
df_fin['disclosed_date'] = pd.to_datetime(df_fin['disclosed_date'])
df_fin = df_fin[df_fin['disclosed_date'] >= '2017-01-01'].copy()

# 必要カラムの確認
print(f"\n財務データのカラム: {df_fin.columns.tolist()}")
print(f"財務データ: {len(df_fin):,} 行")

価格データ読み込み中...


価格データ: 9,148,457 行
財務データ読み込み中...



財務データのカラム: ['disclosed_date', 'disclosed_time', 'code', 'disclosure_number', 'document_type', 'fiscal_quarter', 'CurPerSt', 'CurPerEn', 'CurFYSt', 'fiscal_year_end', 'net_sales', 'operating_profit', 'ordinary_profit', 'net_profit', 'eps', 'DEPS', 'total_assets', 'equity', 'equity_ratio', 'bps', 'FSales', 'FOP', 'FOdP', 'FNP', 'FEPS', 'NxFSales', 'NxFOP', 'NxFOdP', 'NxFEPS', 'DivAnn', 'FDivFY', 'NxFDivFY', 'PayoutRatioAnn', 'ForecastNetSales', 'ForecastOperatingProfit', 'ForecastOrdinaryProfit', 'ForecastProfit', 'ForecastEarningsPerShare']
財務データ: 171,943 行


## 2. データの前処理

In [3]:
# 必須カラムのみ抽出（FNP: 当期純利益予想を含む）
required_cols = ['disclosed_date', 'code', 'bps', 'FNP', 'fiscal_quarter']
df_fin = df_fin[required_cols].copy()

# FNPの欠損値を除外
before_count = len(df_fin)
df_fin = df_fin[df_fin['FNP'].notna()].copy()
print(f"FNP欠損値除外: {before_count:,} → {len(df_fin):,} 行 ({len(df_fin)/before_count*100:.1f}%)")

# BPSの異常値除外
before_count = len(df_fin)
df_fin = df_fin[df_fin['bps'] > 0].copy()
print(f"BPS異常値除外: {before_count:,} → {len(df_fin):,} 行 ({len(df_fin)/before_count*100:.1f}%)")

print(f"\n前処理後の財務データ: {len(df_fin):,} 行")
print(f"対象期間: {df_fin['disclosed_date'].min().date()} ~ {df_fin['disclosed_date'].max().date()}")

FNP欠損値除外: 171,943 → 113,591 行 (66.1%)
BPS異常値除外: 113,591 → 25,892 行 (22.8%)

前処理後の財務データ: 25,892 行
対象期間: 2017-01-04 ~ 2026-01-09


## 3. 価格データのピボット化

In [4]:
print("価格データをピボット化中...")
# adjusted_close（調整後終値）を使用
df_price_pivot = df_price.pivot(index='date', columns='code', values='adjusted_close')
print(f"調整後終値ピボット: {df_price_pivot.shape[0]} 日 × {df_price_pivot.shape[1]} 銘柄")

# volume（出来高）もピボット化
df_volume_pivot = df_price.pivot(index='date', columns='code', values='volume')
print(f"出来高ピボット: {df_volume_pivot.shape[0]} 日 × {df_volume_pivot.shape[1]} 銘柄")

print("ピボット化完了")

価格データをピボット化中...


調整後終値ピボット: 2212 日 × 5226 銘柄


出来高ピボット: 2212 日 × 5226 銘柄
ピボット化完了


## 4. リバランス日生成（月次）

In [5]:
# 月末営業日
trading_days_df = pd.DataFrame({'date': df_price_pivot.index})
trading_days_df['year'] = trading_days_df['date'].dt.year
trading_days_df['month'] = trading_days_df['date'].dt.month
rebalance_dates = trading_days_df.groupby(['year', 'month'])['date'].max().values
rebalance_dates = pd.Series(rebalance_dates).sort_values().reset_index(drop=True)

print(f"リバランス日数: {len(rebalance_dates)}")
print(f"期間: {rebalance_dates.iloc[0].date()} ~ {rebalance_dates.iloc[-1].date()}")

リバランス日数: 109
期間: 2017-01-31 ~ 2026-01-22


## 5. 財務データの事前処理（各日の最新データ + 予想リビジョン計算）

In [6]:
print("財務データの事前処理中（予想リビジョン計算）...")

# 各リバランス日での利用可能な財務データ + 予想リビジョン
fin_by_date = {}

for i, rdate in enumerate(rebalance_dates):
    if i % 12 == 0:
        print(f"  進捗: {i}/{len(rebalance_dates)}")
    
    # その日までに開示された財務データ
    available = df_fin[df_fin['disclosed_date'] <= rdate].copy()
    
    # 各銘柄の最新データ
    latest = available.sort_values('disclosed_date').groupby('code').tail(1)
    latest = latest.set_index('code')[['bps', 'FNP']]
    
    # 予想リビジョン計算（1ヶ月前との比較）
    # 前回リバランス日の予想データを取得
    if i > 0:
        prev_rdate = rebalance_dates.iloc[i-1]
        prev_available = df_fin[df_fin['disclosed_date'] <= prev_rdate].copy()
        prev_latest = prev_available.sort_values('disclosed_date').groupby('code').tail(1)
        prev_latest = prev_latest.set_index('code')[['FNP']]
        prev_latest.columns = ['FNP_prev']
        
        # マージしてリビジョン計算
        merged = latest.join(prev_latest, how='left')
        merged['revision'] = ((merged['FNP'] - merged['FNP_prev']) / merged['FNP_prev']) * 100
        latest['revision'] = merged['revision']
    else:
        # 初回はリビジョンなし
        latest['revision'] = np.nan
    
    fin_by_date[rdate] = latest

print("財務データ事前処理完了")

財務データの事前処理中（予想リビジョン計算）...
  進捗: 0/109


  進捗: 12/109


  進捗: 24/109


  進捗: 36/109


  進捗: 48/109


  進捗: 60/109


  進捗: 72/109


  進捗: 84/109


  進捗: 96/109


  進捗: 108/109
財務データ事前処理完了


## 6. 銘柄選定関数（4つのポートフォリオ）

In [7]:
def screen_stocks_value_trap(
    rebalance_date, 
    prices_pivot, 
    volume_pivot, 
    fin_data, 
    n_stocks=20, 
    filter_type='baseline'
):
    """
    バリュートラップ回避版銘柄スクリーニング
    
    filter_type:
    - 'baseline': フィルタなし（低PBRのみ）
    - 'revision': 低PBR × 予想リビジョン > 0
    - 'volume': 低PBR × 出来高 > 中央値
    - 'both': 低PBR × 予想リビジョン > 0 × 出来高 > 中央値
    """
    # その日の価格（調整後終値）
    if rebalance_date not in prices_pivot.index:
        return pd.DataFrame()
    
    prices = prices_pivot.loc[rebalance_date].dropna()
    
    # 出来高
    if rebalance_date not in volume_pivot.index:
        return pd.DataFrame()
    
    volumes = volume_pivot.loc[rebalance_date].dropna()
    
    # 財務データ
    if rebalance_date not in fin_data:
        return pd.DataFrame()
    
    fin = fin_data[rebalance_date]
    
    # マージ
    merged = pd.DataFrame({
        'adjusted_close': prices,
        'volume': volumes,
        'bps': fin['bps'],
        'FNP': fin['FNP'],
        'revision': fin['revision']
    }).dropna(subset=['adjusted_close', 'bps'])  # revision, volumeは後でフィルタ
    
    if len(merged) < n_stocks:
        return pd.DataFrame()
    
    # PBR計算
    merged['pbr'] = merged['adjusted_close'] / merged['bps']
    
    # PBR異常値除外
    merged = merged[(merged['pbr'] > 0) & (merged['pbr'] < 50)]
    
    if len(merged) < n_stocks:
        return pd.DataFrame()
    
    # フィルタ適用
    if filter_type == 'baseline':
        # フィルタなし（低PBRのみ）
        candidates = merged
    
    elif filter_type == 'revision':
        # 予想リビジョン > 0のみ
        candidates = merged[merged['revision'] > 0]
    
    elif filter_type == 'volume':
        # 出来高 > 中央値
        volume_median = merged['volume'].median()
        candidates = merged[merged['volume'] > volume_median]
    
    elif filter_type == 'both':
        # 両方適用
        volume_median = merged['volume'].median()
        candidates = merged[
            (merged['revision'] > 0) & 
            (merged['volume'] > volume_median)
        ]
    
    else:
        raise ValueError(f"Unknown filter_type: {filter_type}")
    
    # 銘柄数が足りない場合はフォールバック（PBR下位）
    if len(candidates) < n_stocks:
        candidates = merged.nsmallest(n_stocks, 'pbr')
    
    # PBR下位から選定
    selected = candidates.nsmallest(n_stocks, 'pbr')
    
    return selected.reset_index()

# テスト
test_date = rebalance_dates.iloc[10]
test_baseline = screen_stocks_value_trap(test_date, df_price_pivot, df_volume_pivot, fin_by_date, filter_type='baseline')
test_revision = screen_stocks_value_trap(test_date, df_price_pivot, df_volume_pivot, fin_by_date, filter_type='revision')
print(f"テスト ({test_date.date()}):")
print(f"  ベースライン: {len(test_baseline)} 銘柄")
print(f"  リビジョンフィルタ: {len(test_revision)} 銘柄")

テスト (2017-11-30):
  ベースライン: 20 銘柄
  リビジョンフィルタ: 20 銘柄


## 7. バックテスト関数

In [8]:
def run_backtest(
    rebalance_dates,
    prices_pivot,
    volume_pivot,
    fin_data,
    filter_type='baseline',
    initial_cash=10_000_000,
    n_stocks=20,
    tax_rate=0.20315,
    unit=100
):
    """
    バックテスト実行
    """
    cash = initial_cash
    portfolio = {}
    annual_realized_pnl = 0
    current_year = None
    results = []
    
    print(f"バックテスト開始: {filter_type}")
    print(f"初期資本: {initial_cash:,}円")
    print(f"リバランス回数: {len(rebalance_dates)}")
    print()
    
    for i, rebalance_date in enumerate(rebalance_dates):
        if i % 12 == 0:
            print(f"  進捗: {i}/{len(rebalance_dates)} - {rebalance_date.date()}")
        
        # 年の切り替わり
        if current_year != rebalance_date.year:
            if current_year is not None and annual_realized_pnl > 0:
                tax = annual_realized_pnl * tax_rate
                cash -= tax
            annual_realized_pnl = 0
            current_year = rebalance_date.year
        
        # 既存ポートフォリオ売却
        sell_value = 0
        if rebalance_date in prices_pivot.index:
            for code, position in portfolio.items():
                if code in prices_pivot.columns:
                    sell_price = prices_pivot.loc[rebalance_date, code]
                    if pd.notna(sell_price):
                        sell_amount = position['shares'] * sell_price
                        sell_value += sell_amount
                        pnl = (sell_price - position['buy_price']) * position['shares']
                        annual_realized_pnl += pnl
        
        cash += sell_value
        portfolio = {}
        
        # 銘柄選定
        selected = screen_stocks_value_trap(
            rebalance_date, 
            prices_pivot, 
            volume_pivot, 
            fin_data, 
            n_stocks, 
            filter_type
        )
        
        if len(selected) == 0:
            results.append({
                'date': rebalance_date,
                'cash': cash,
                'n_stocks': 0,
                'invested': 0,
                'annual_pnl': annual_realized_pnl,
                'portfolio': {}
            })
            continue
        
        # 購入
        target_per_stock = cash / len(selected)
        total_invested = 0
        
        for _, row in selected.iterrows():
            code = row['code']
            price = row['adjusted_close']
            shares = int(target_per_stock / (price * unit)) * unit
            
            if shares > 0:
                invest_amount = shares * price
                total_invested += invest_amount
                portfolio[code] = {'shares': shares, 'buy_price': price}
        
        cash -= total_invested
        
        results.append({
            'date': rebalance_date,
            'cash': cash,
            'n_stocks': len(portfolio),
            'invested': total_invested,
            'annual_pnl': annual_realized_pnl,
            'portfolio': portfolio.copy()
        })
    
    # 最終税金
    if annual_realized_pnl > 0:
        tax = annual_realized_pnl * tax_rate
        cash -= tax
        results[-1]['cash'] = cash
    
    # 最終日時点での保有株式を時価評価
    final_date = prices_pivot.index.max()
    final_portfolio_value = 0
    
    if len(portfolio) > 0:
        for code, position in portfolio.items():
            if code in prices_pivot.columns:
                final_price = prices_pivot.loc[final_date, code]
                if pd.notna(final_price):
                    final_portfolio_value += position['shares'] * final_price
        results[-1]['invested'] = final_portfolio_value
    
    print(f"\nバックテスト完了: {filter_type}")
    print(f"最終現金: {cash:,.0f}円")
    print(f"最終保有株式時価: {final_portfolio_value:,.0f}円")
    print(f"最終総資産: {(cash + final_portfolio_value):,.0f}円")
    print()
    
    return results

print("バックテスト関数定義完了")

バックテスト関数定義完了


## 8. 4つのポートフォリオのバックテスト実行

In [9]:
# 4つのポートフォリオ
portfolios = {
    'baseline': 'ベースライン（フィルタなし）',
    'revision': 'フィルタA（予想リビジョン）',
    'volume': 'フィルタB（出来高）',
    'both': 'フィルタC（両方）'
}

all_results = {}

for filter_type, description in portfolios.items():
    print(f"\n{'='*60}")
    print(f"実行中: {description}")
    print(f"{'='*60}")
    results = run_backtest(
        rebalance_dates,
        df_price_pivot,
        df_volume_pivot,
        fin_by_date,
        filter_type=filter_type
    )
    all_results[filter_type] = results

print("\n全ポートフォリオのバックテスト完了")


実行中: ベースライン（フィルタなし）
バックテスト開始: baseline
初期資本: 10,000,000円
リバランス回数: 109

  進捗: 0/109 - 2017-01-31


  進捗: 12/109 - 2018-01-31


  進捗: 24/109 - 2019-01-31


  進捗: 36/109 - 2020-01-31


  進捗: 48/109 - 2021-01-29


...

  進捗: 60/109 - 2022-01-31


  進捗: 72/109 - 2023-01-31


  進捗: 84/109 - 2024-01-31


  進捗: 96/109 - 2025-01-31


  進捗: 108/109 - 2026-01-22

バックテスト完了: both
最終現金: 508,199円
最終保有株式時価: 3,478,300円
最終総資産: 3,986,499円


全ポートフォリオのバックテスト完了


## 9. パフォーマンス分析

In [10]:
def calculate_metrics(results, initial_cash=10_000_000):
    """
    パフォーマンス指標を計算
    """
    df = pd.DataFrame(results)
    df['total_value'] = df['cash'] + df['invested']
    df['return'] = df['total_value'].pct_change()
    df['cumulative_return'] = (1 + df['return']).cumprod() - 1
    
    # 総リターン
    total_return = df['cumulative_return'].iloc[-1]
    
    # 年率換算
    years = (df['date'].iloc[-1] - df['date'].iloc[0]).days / 365.25
    annual_return = (1 + total_return) ** (1 / years) - 1
    
    # ボラティリティ
    annual_volatility = df['return'].std() * np.sqrt(12)
    
    # 最大ドローダウン
    df['peak'] = df['total_value'].cummax()
    df['drawdown'] = (df['total_value'] - df['peak']) / df['peak']
    mdd = df['drawdown'].min()
    
    # シャープレシオ
    sharpe = df['return'].mean() / df['return'].std() * np.sqrt(12) if df['return'].std() > 0 else 0
    
    # カルマー比
    calmar = annual_return / abs(mdd) if mdd != 0 else 0
    
    return {
        'total_return': total_return,
        'annual_return': annual_return,
        'annual_volatility': annual_volatility,
        'mdd': mdd,
        'sharpe': sharpe,
        'calmar': calmar,
        'final_value': df['total_value'].iloc[-1],
        'df': df
    }

# 全ポートフォリオの指標計算
all_metrics = {}
for filter_type in portfolios.keys():
    all_metrics[filter_type] = calculate_metrics(all_results[filter_type])

print("パフォーマンス指標計算完了")

パフォーマンス指標計算完了


## 10. 結果の比較表

In [11]:
# 比較表作成
comparison_data = []

for filter_type, description in portfolios.items():
    metrics = all_metrics[filter_type]
    comparison_data.append({
        'ポートフォリオ': description,
        '最終資産（円）': f"{metrics['final_value']:,.0f}",
        '総リターン（%）': f"{metrics['total_return']*100:.2f}",
        '年率リターン（%）': f"{metrics['annual_return']*100:.2f}",
        '年率ボラ（%）': f"{metrics['annual_volatility']*100:.2f}",
        'MDD（%）': f"{metrics['mdd']*100:.2f}",
        'シャープ': f"{metrics['sharpe']:.2f}",
        'カルマー': f"{metrics['calmar']:.2f}"
    })

df_comparison = pd.DataFrame(comparison_data)
print("\n4つのポートフォリオ比較")
print("="*100)
display(df_comparison)


4つのポートフォリオ比較


,ポートフォリオ,最終資産（円）,総リターン（%）,年率リターン（%）,年率ボラ（%）,MDD（%）,シャープ,カルマー
0,ベースライン（フィルタなし）,"2,322,200",-76.78,-15.01,18.32,-87.02,-0.78,-0.17
1,フィルタA（予想リビジョン）,"2,691,644",-73.08,-13.60,18.09,-81.24,-0.70,-0.17
2,フィルタB（出来高）,"36,105,562",261.06,15.38,18.84,-28.74,0.85,0.54
3,フィルタC（両方）,"3,986,499",-60.14,-9.74,18.31,-71.64,-0.46,-0.14


## 11. フィルタ有効性の分析

In [12]:
# ベースラインとの比較
baseline_metrics = all_metrics['baseline']

effectiveness_data = []
for filter_type, description in portfolios.items():
    if filter_type == 'baseline':
        continue
    
    metrics = all_metrics[filter_type]
    
    # 改善率計算
    return_improvement = (metrics['annual_return'] - baseline_metrics['annual_return']) * 100
    mdd_improvement = (baseline_metrics['mdd'] - metrics['mdd']) * 100  # MDDは小さい方が良い
    sharpe_improvement = metrics['sharpe'] - baseline_metrics['sharpe']
    
    effectiveness_data.append({
        'フィルタ': description,
        '年率リターン改善（%pt）': f"{return_improvement:+.2f}",
        'MDD改善（%pt）': f"{mdd_improvement:+.2f}",
        'シャープ改善': f"{sharpe_improvement:+.2f}",
        '最終資産差（円）': f"{(metrics['final_value'] - baseline_metrics['final_value']):+,.0f}"
    })

df_effectiveness = pd.DataFrame(effectiveness_data)
print("\nフィルタ有効性分析（ベースラインとの比較）")
print("="*100)
display(df_effectiveness)


フィルタ有効性分析（ベースラインとの比較）


,フィルタ,年率リターン改善（%pt）,MDD改善（%pt）,シャープ改善,最終資産差（円）
0,フィルタA（予想リビジョン）,+1.41,-5.78,+0.08,"+369,444"
1,フィルタB（出来高）,+30.39,-58.27,+1.64,"+33,783,362"
2,フィルタC（両方）,+5.27,-15.38,+0.32,"+1,664,298"


## 12. 結果の保存

In [13]:
# 1. バックテスト結果CSV（全ポートフォリオ統合）
output_dir = PROJECT_ROOT / 'analyses/20260221_0900_quants_model_value_trap_avoidance'

# 全ポートフォリオの日次データを結合
combined_results = []
for filter_type, description in portfolios.items():
    df = all_metrics[filter_type]['df'].copy()
    df['portfolio'] = description
    combined_results.append(df[['date', 'portfolio', 'total_value', 'cumulative_return', 'drawdown']])

df_combined = pd.concat(combined_results, ignore_index=True)
csv_path = output_dir / 'backtest_results.csv'
df_combined.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f"結果を保存: {csv_path}")

# 2. 評価指標JSON
metrics_dict = {}
for filter_type, description in portfolios.items():
    metrics = all_metrics[filter_type]
    metrics_dict[filter_type] = {
        'description': description,
        'total_return': float(metrics['total_return']),
        'annual_return': float(metrics['annual_return']),
        'annual_volatility': float(metrics['annual_volatility']),
        'mdd': float(metrics['mdd']),
        'sharpe': float(metrics['sharpe']),
        'calmar': float(metrics['calmar']),
        'final_value': float(metrics['final_value'])
    }

json_path = output_dir / 'backtest_metrics.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(metrics_dict, f, indent=2, ensure_ascii=False)
print(f"指標を保存: {json_path}")

# 3. サマリテキスト
summary_path = output_dir / 'performance_summary.txt'
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write("戦略4: バリュートラップ回避モデル - バックテスト結果\n")
    f.write("="*80 + "\n")
    f.write(f"期間: {rebalance_dates.iloc[0].date()} ~ {rebalance_dates.iloc[-1].date()}\n")
    f.write(f"初期資本: 10,000,000円\n")
    f.write(f"リバランス頻度: 月次\n")
    f.write(f"保有銘柄数: 20\n")
    f.write("\n")
    
    for filter_type, description in portfolios.items():
        metrics = all_metrics[filter_type]
        f.write(f"\n【{description}】\n")
        f.write(f"  最終資産: {metrics['final_value']:,.0f}円\n")
        f.write(f"  総リターン: {metrics['total_return']*100:.2f}%\n")
        f.write(f"  年率リターン: {metrics['annual_return']*100:.2f}%\n")
        f.write(f"  年率ボラティリティ: {metrics['annual_volatility']*100:.2f}%\n")
        f.write(f"  最大ドローダウン: {metrics['mdd']*100:.2f}%\n")
        f.write(f"  シャープレシオ: {metrics['sharpe']:.2f}\n")
        f.write(f"  カルマー比: {metrics['calmar']:.2f}\n")
    
    f.write("\n" + "="*80 + "\n")
    f.write("\nフィルタ有効性（ベースラインとの比較）\n")
    f.write("="*80 + "\n")
    for _, row in df_effectiveness.iterrows():
        f.write(f"\n【{row['フィルタ']}】\n")
        f.write(f"  年率リターン改善: {row['年率リターン改善（%pt）']}%pt\n")
        f.write(f"  MDD改善: {row['MDD改善（%pt）']}%pt\n")
        f.write(f"  シャープ改善: {row['シャープ改善']}\n")
        f.write(f"  最終資産差: {row['最終資産差（円）']}円\n")

print(f"サマリを保存: {summary_path}")

# 4. フィルタ有効性CSV
effectiveness_path = output_dir / 'filter_effectiveness.csv'
df_effectiveness.to_csv(effectiveness_path, index=False, encoding='utf-8-sig')
print(f"フィルタ有効性を保存: {effectiveness_path}")

print("\n全ての成果物を保存完了")

結果を保存: C:\Users\yongr\claude project\workspace\analyses\20260221_0900_quants_model_value_trap_avoidance\backtest_results.csv


指標を保存: C:\Users\yongr\claude project\workspace\analyses\20260221_0900_quants_model_value_trap_avoidance\backtest_metrics.json
サマリを保存: C:\Users\yongr\claude project\workspace\analyses\20260221_0900_quants_model_value_trap_avoidance\performance_summary.txt
フィルタ有効性を保存: C:\Users\yongr\claude project\workspace\analyses\20260221_0900_quants_model_value_trap_avoidance\filter_effectiveness.csv

全ての成果物を保存完了


## 13. 結論

In [14]:
print("\n" + "="*100)
print("結論")
print("="*100)

# 最も優れたフィルタを特定
best_filter = None
best_sharpe = -999

for filter_type in ['revision', 'volume', 'both']:
    if all_metrics[filter_type]['sharpe'] > best_sharpe:
        best_sharpe = all_metrics[filter_type]['sharpe']
        best_filter = filter_type

best_description = portfolios[best_filter]
print(f"\n最も優れたフィルタ: {best_description}")
print(f"  シャープレシオ: {all_metrics[best_filter]['sharpe']:.2f}")
print(f"  年率リターン: {all_metrics[best_filter]['annual_return']*100:.2f}%")
print(f"  最大ドローダウン: {all_metrics[best_filter]['mdd']*100:.2f}%")

# ベースラインとの比較
baseline_final = baseline_metrics['final_value']
best_final = all_metrics[best_filter]['final_value']
improvement_pct = (best_final / baseline_final - 1) * 100

print(f"\nベースライン（フィルタなし）との比較:")
print(f"  ベースライン最終資産: {baseline_final:,.0f}円")
print(f"  {best_description}最終資産: {best_final:,.0f}円")
print(f"  改善率: {improvement_pct:+.2f}%")

print("\n" + "="*100)


結論

最も優れたフィルタ: フィルタB（出来高）
  シャープレシオ: 0.85
  年率リターン: 15.38%
  最大ドローダウン: -28.74%

ベースライン（フィルタなし）との比較:
  ベースライン最終資産: 2,322,200円
  フィルタB（出来高）最終資産: 36,105,562円
  改善率: +1454.80%

